In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [28]:
df1 = pd.read_csv('1.csv', sep=';')
df2 = pd.read_csv('10.csv', sep=';')
df3 = pd.read_csv('100.csv', sep=';')

df = pd.concat(
    [df1, df2, df3],
    ignore_index=True
)

print(df.shape)

(45029, 11)


In [29]:
print(df.columns.tolist())

['Timestamp [ms]', '\tCPU cores', '\tCPU capacity provisioned [MHZ]', '\tCPU usage [MHZ]', '\tCPU usage [%]', '\tMemory capacity provisioned [KB]', '\tMemory usage [KB]', '\tDisk read throughput [KB/s]', '\tDisk write throughput [KB/s]', '\tNetwork received throughput [KB/s]', '\tNetwork transmitted throughput [KB/s]']


In [30]:
features = [
    "CPU capacity provisioned [MHZ]",
    "CPU usage [MHZ]",
    "CPU usage [%]",
    "Memory capacity provisioned [KB]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
    "Network received throughput [KB/s]",
    "Network transmitted throughput [KB/s]"
]

# Clean column names by stripping leading/trailing whitespace
df.columns = df.columns.str.strip()

data = df[features]

In [31]:
data.head()

,CPU capacity provisioned [MHZ],CPU usage [MHZ],CPU usage [%],Memory capacity provisioned [KB],Memory usage [KB],Disk read throughput [KB/s],Disk write throughput [KB/s],Network received throughput [KB/s],Network transmitted throughput [KB/s]
0,11703.99824,10912.027692,93.233333,67108864.0,6.129274e+06,0.133333,15981.600000,0.000000,2.133333
1,11703.99824,10890.570362,93.050000,67108864.0,6.755624e+06,1.333333,19137.333333,0.000000,2.600000
2,11703.99824,10434.114431,89.150000,67108864.0,8.947846e+06,2.533333,19974.933333,535.666667,23.933333
3,11703.99824,10539.450415,90.050000,67108864.0,1.879048e+07,5.466667,8791.800000,349.666667,5.466667
4,11703.99824,10951.041020,93.566667,67108864.0,9.305761e+06,5.400000,15679.533333,0.000000,2.066667


In [32]:
data = data.replace(
    [np.inf, -np.inf],
    np.nan
)

data = data.interpolate()

data = data.dropna()

print(data.shape)

(45029, 9)


In [33]:
print(data.describe())

       CPU capacity provisioned [MHZ]  CPU usage [MHZ]  CPU usage [%]  \
count                    45029.000000     45029.000000   45029.000000   
mean                      2711.099326       106.975802       0.986016   
std                       4718.454865       936.695624       8.008164   
min                          0.000000         0.000000       0.000000   
25%                          0.000000         0.000000       0.000000   
50%                          0.000000         0.000000       0.000000   
75%                       5199.998600        60.470658       0.516667   
max                      11704.000484     11454.312944      97.866667   

       Memory capacity provisioned [KB]  Memory usage [KB]  \
count                      4.502900e+04       4.502900e+04   
mean                       1.358440e+07       1.485989e+05   
std                        2.620199e+07       9.626736e+05   
min                        0.000000e+00       0.000000e+00   
25%                        0.000

In [34]:
for col in features:
    print(col)
    print((data[col] == 0).mean() * 100)
    print()

CPU capacity provisioned [MHZ]
73.35716982389127

CPU usage [MHZ]
73.35716982389127

CPU usage [%]
73.35716982389127

Memory capacity provisioned [KB]
73.34384507761665

Memory usage [KB]
90.31957183148637

Disk read throughput [KB/s]
98.67640853672079

Disk write throughput [KB/s]
74.23438228697063

Network received throughput [KB/s]
98.84963023829087

Network transmitted throughput [KB/s]
76.39743276555109



In [9]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import *

inputs = Input(
    shape=(20,9)
)

x = LSTM(
    128,
    activation='relu',
    return_sequences=True
)(inputs)

x = LSTM(
    64,
    activation='relu',
    return_sequences=False
)(x)

x = RepeatVector(20)(x)

x = LSTM(
    64,
    activation='relu',
    return_sequences=True
)(x)

x = LSTM(
    128,
    activation='relu',
    return_sequences=True
)(x)

outputs = TimeDistributed(
    Dense(9)
)(x)

model = Model(
    inputs,
    outputs
)

model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 9)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 20, 128)        │        70,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 20, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 20, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 9)          │         1,161 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 253,065 (988.54 KB)

 Trainable params: 253,065 (988.54 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    X_train,
    X_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    shuffle=False
)

Epoch 1/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - loss: 0.0135 - val_loss: 3.8415e-05
Epoch 2/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.5790 - val_loss: 3.1062e-04
Epoch 3/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0090 - val_loss: 4.2471e-05
Epoch 4/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0034 - val_loss: 2.4825e-06
Epoch 5/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0032 - val_loss: 8.4610e-07
Epoch 6/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0028 - val_loss: 4.3808e-07
Epoch 7/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0024 - val_loss: 4.4961e-07
Epoch 8/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0017 - val_loss: 2.8965e-07
Epoch 9/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0028 - val_loss: 3.3127e-07
Epoch 10/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0018 - val_loss: 2.4526e-07
Epoch 11/50
507/507 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0014 - val_loss: 1.9430e

In [17]:
train_pred = model.predict(X_train)

train_loss = np.mean(
    np.square(
        X_train - train_pred
    ),
    axis=(1,2)
)

threshold = np.percentile(train_loss, 99)

print(threshold)

1126/1126 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
0.04230576338873525


In [20]:
test_pred = model.predict(X_test)

282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [21]:
test_loss = np.mean(
    np.square(
        X_test - test_pred
    ),
    axis=(1,2)
)

In [22]:
print(X_test.shape)
print(test_pred.shape)

(9002, 20, 9)
(9002, 20, 9)


In [23]:
print(test_loss[:20])

[2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08
 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08
 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08
 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08 2.9907303e-08]


In [24]:
print("DF Shape:", df.shape)

print("X Shape:", X.shape)

print("X_train Shape:", X_train.shape)

print("X_test Shape:", X_test.shape)

DF Shape: (45029, 11)
X Shape: (45009, 20, 9)
X_train Shape: (36007, 20, 9)
X_test Shape: (9002, 20, 9)


In [25]:
print("First sequence")

print(X_test[0])

print("\nSecond sequence")

print(X_test[1])

First sequence
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Second sequence
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0.

In [26]:
print(
    np.array_equal(
        X_test[0],
        X_test[1]
    )
)

True
